<a href="https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/12-case-studies-and-capstone/04-deepfake-image-detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Case Study D — Deepfake detection for EU AI Act Art. 50(4): detector, VLM, or both?

**Goal:** Build a test system that compares four ways to flag **undisclosed deepfakes**, the thing Article 50(4) of the EU AI Act actually regulates, and let the numbers pick the design: a traditional detector built by transfer learning, a vision-language model (VLM) end to end, a provenance check, and a hybrid of all three.

Part of [ai-engineer-notebooks](https://github.com/calmrocks/ai-engineer-notebooks), the hands-on companion to the [FDE / AI Engineer transition plan](https://www.calm.rocks/resources/career-development/transition-fde-ai-engineer/).

> **The thesis, up front:** "is this image AI-generated?" and "does this image break Art. 50(4)?" are different questions. The first is perception, where a trained detector should beat a general model. The second also needs *meaning* (does it depict a real person or event? is it evidently satire?) and *provenance* (was it already disclosed?), which no pixel classifier answers. So the likely winner is a hybrid, but "likely" isn't a design decision: this case builds the eval first ([02](../02-evals-basics/01-measuring-outputs.ipynb), [04](../04-evals/01-golden-sets.ipynb)) and measures it.

Multimodal stays off the core spine ([curriculum decision #3](../docs/curriculum-plan.md)); this is a case study that composes the evals spine with one new modality. It also leaves the repo's "raw API, no frameworks" rule for one part on purpose: the traditional detector is classical ML, so it uses `transformers` + `scikit-learn`, the tools you'd actually reach for. The transfer-learning fine-tune runs in a **fenced GPU appendix**, like [06](../06-adaptation/01-fine-tune-vs-rag-vs-prompt.ipynb) and [09](../09-serving-inference/01-serving-frameworks.ipynb).

## Setup

Self-contained. Get a free key at [console.groq.com](https://console.groq.com/); in Colab add it via the **key icon** → secret `GROQ_API_KEY`. (Full walkthrough: [00-setup](../00-setup/00-environment.ipynb).)

Two Groq models: the default text model labels the eval set, and an **image-capable** model plays the VLM. Groq's vision docs list `qwen/qwen3.8-27b` (up to 3 images and 20 MB per request, 2,048 tokens per image, as of 2026-10). If calls 404, check [console.groq.com/docs/vision](https://console.groq.com/docs/vision).

The main path runs on a CPU runtime. Only the appendix needs a GPU.

**How to run it:** the defaults are deliberately tiny (15 test images, original variant only, ~17 VLM calls), so a first run goes end to end in a few minutes and shows every step working, starting with **one image walked through step by step**. Once that works, raise the size knobs in the data cell (`N_TEST_PER_GROUP`, `N_TRAIN_PER_CLASS`, `N_CAL_PER_CLASS`, and `VARIANTS` for the laundering experiment) and rerun for numbers you can trust.

In [ ]:
%pip install -q "git+https://github.com/calmrocks/ai-engineer-notebooks.git@v0.1.0"

In [ ]:
from aien import setup
client, TEXT_MODEL = setup()
VISION_MODEL = "qwen/qwen3.8-27b"

## The ask, and what the law actually says

A trust-and-safety team: *"Article 50 applies from 2 August 2026. Find the deepfakes on our platform that should have been labelled and weren't."* Before building anything, read the article ([11/01 scoping](../11-customer-craft/01-scoping-and-discovery.ipynb)). Article 50 splits the duties:

| Paragraph | Who | Duty |
|---|---|---|
| 50(2) | **providers** of generative systems | mark outputs in a machine-readable, detectable way |
| 50(3) | deployers of emotion recognition / biometric categorisation | inform the people exposed |
| **50(4)** | **deployers** of deepfakes | disclose that the content is generated or manipulated; evidently artistic, satirical or fictional works only need a limited disclosure |

A **deepfake** (Art. 3(60)) is generated or manipulated content that resembles existing persons, objects, places, entities or events and would falsely appear authentic. So an image is a 50(4) problem only when **all three** hold:

1. it is AI-generated or AI-manipulated (*perception*);
2. it depicts a real, identifiable entity or event, and is not evidently fiction or satire (*meaning*);
3. the published file carries **no disclosure**: no visible label, no machine-readable marking (*provenance*).

The Commission's June 2026 Code of Practice on marking and labelling asks providers to layer at least two machine-readable marks (e.g. metadata plus watermark), so condition 3 is partly checkable from the file itself.

**Scope for this case:** still images only (50(4) also covers audio and video). The system **flags for human review**; deciding that something is a violation is a legal call, and nothing here is legal advice.

## Data

**[OpenFake](https://huggingface.co/datasets/ComplexDataLab/OpenFake)** (`core` subset) fits this case unusually well:

- real and generated images, with the **generator name** (75 sources) and, for most generators, its **release date**, so we can hold out the *newest* generators as "unseen", the way a deployed detector meets them (undated generators are older ones and count as seen);
- the **prompt / caption** behind each image, which we use to label condition 2 (real entity? evident fiction?);
- many prompts name real public figures and events, which is exactly the 50(4) territory.

> **⚠️ Production reality —** OpenFake is **CC BY-NC 4.0** (non-commercial) and contains images of real public figures. To survive network failures, `sample()` checkpoints the **downscaled** images it picks under `SAMPLE_DIR`; keep that folder private, don't redistribute it, and delete it when you're done. The model-output cache (`CACHE_PATH`) holds scores only. Check the dataset card for any terms beyond the license tag, and if the dataset is gated, accept its terms on Hugging Face and add an `HF_TOKEN` secret.

Three splits, with distinct jobs: **train** (fit the detector), **calibration** (pick its threshold), **test** (report). Generators released on or after `HOLDOUT_AFTER` never appear in train or calibration.

In [ ]:
import base64, hashlib, io, json, os, random, re, time, urllib.parse, urllib.request
from concurrent.futures import ThreadPoolExecutor
from PIL import Image, ImageDraw, ImageFont
from PIL.PngImagePlugin import PngInfo

SEED = 0
HOLDOUT_AFTER = "2025-06-01"      # fakes from generators released on/after this are "unseen"

# --- Size knobs. Start tiny to get the pipeline working end to end, then raise them for real numbers. ---
N_TRAIN_PER_CLASS = 100           # detector training images per class (400+ for real results)
N_CAL_PER_CLASS = 50              # threshold calibration images per class (100+ for real results)
N_TEST_PER_GROUP = 5              # test images per group: real / fake_seen / fake_unseen (30+ for real results)
VARIANTS = ["original"]           # add "laundered" for the robustness experiment (doubles the VLM calls)

TARGET_FPR = 0.05                 # operating point: flag at most 5% of real images
MAX_SIDE = 512                    # images are downscaled once on load
MAX_TOKENS = 1024                 # headroom for models that reason before answering
VLM_TOKENS_PER_DAY = 200_000      # the vision model's daily token cap on your Groq plan (console.groq.com/settings/limits)
CACHE_PATH = "case_d_cache.json"  # model outputs only; this is what lets a run span several days
SAMPLE_DIR = "case_d_samples"     # resumable sample checkpoints (downscaled images); see the license note
# A free Colab runtime's disk is wiped on restart. To resume on another day, keep both on Drive:
#   from google.colab import drive; drive.mount("/content/drive")
#   CACHE_PATH = "/content/drive/MyDrive/case_d_cache.json"
#   SAMPLE_DIR = "/content/drive/MyDrive/case_d_samples"

N_TEST = {g: N_TEST_PER_GROUP for g in ("real", "fake_seen", "fake_unseen")}
DATASET_ID, SUBSET = "ComplexDataLab/OpenFake", "core"
VIEWER = "https://datasets-server.huggingface.co"      # Hugging Face's dataset viewer API

def hf_token():
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"]
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:                               # not in Colab, or no HF_TOKEN secret: anonymous is fine
        return None

HEADERS = {"Authorization": f"Bearer {hf_token()}"} if hf_token() else {}

def ram():
    try:
        import psutil
        return f"{psutil.Process().memory_info().rss / 1e9:.1f} GB RAM in use"
    except ImportError:
        return "RAM n/a"

def to_png(raw, max_side):
    img = Image.open(io.BytesIO(raw))
    img.draft(img.mode, (max_side, max_side))   # JPEGs decode straight at reduced size
    img = img.convert("RGB")
    img.thumbnail((max_side, max_side))
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return buf.getvalue()

def group_of(label, release_date):
    if label == "real":
        return "real"
    d = str(release_date or "")[:10]
    d = d + "-01" if len(d) == 7 else d            # "2025-04" → "2025-04-01"
    # Undated ("unknown") generators are the older ones, so they count as seen.
    return "fake_unseen" if d[:1].isdigit() and d >= HOLDOUT_AFTER else "fake_seen"

def retry(fn, what, tries=4):
    """The Hub times out or drops connections under load: retry with backoff, then give up on this piece."""
    for attempt in range(1, tries + 1):
        try:
            return fn()
        except Exception as e:
            print(f"   {what}: {type(e).__name__}, attempt {attempt}/{tries}")
            if attempt < tries:
                time.sleep(5 * attempt)
    return None

def http_get(url, what, tries=4):
    def fetch():
        with urllib.request.urlopen(urllib.request.Request(url, headers=HEADERS), timeout=60) as r:
            return r.read()
    return retry(fetch, what, tries)

def rows_page(split, offset, length=100):
    q = urllib.parse.urlencode({"dataset": DATASET_ID, "config": SUBSET, "split": split,
                                "offset": offset, "length": length})
    body = http_get(f"{VIEWER}/rows?{q}", f"{split} rows @{offset}")
    return json.loads(body) if body else None

def write_atomic(path, data):
    with open(path + ".tmp", "wb") as fh:
        fh.write(data)
    os.replace(path + ".tmp", path)                 # a crash mid-write never leaves a half file

def plan(split, quotas, rng, pool, max_pages):
    """Pass 1, metadata only: read pages of 100 rows at random offsets (each row carries its label,
    generator, date, prompt and a link to its image), then pick round-robin across generators."""
    first = rows_page(split, 0, 1)
    if first is None:
        raise RuntimeError(f"{split}: the dataset viewer API isn't answering; rerun this cell in a few minutes")
    offsets = list(range(0, first["num_rows_total"], 100))
    rng.shuffle(offsets)
    cands, pages = {g: [] for g in quotas}, 0
    for off in offsets:
        if pages >= max_pages or all(len(cands[g]) >= pool * q for g, q in quotas.items()):
            break
        page = rows_page(split, off)
        if page is None:
            continue
        pages += 1
        for x in page["rows"]:
            r = x["row"]
            label = str(r["label"]).lower()
            g = group_of(label, r.get("release_date"))
            if g in cands:
                cands[g].append({"row_idx": x["row_idx"], "group": g, "label": label, "model": r.get("model"),
                                 "release_date": str(r.get("release_date") or "")[:10],
                                 "prompt": r.get("prompt") or "", "src": r["image"]["src"]})

    picked = []
    for g, q in quotas.items():
        by_gen = {}
        for c in cands[g]:
            by_gen.setdefault(c["model"], []).append(c)
        buckets = list(by_gen.values())
        for b in buckets:
            rng.shuffle(b)
        rng.shuffle(buckets)
        chosen = []
        while len(chosen) < q and any(buckets):
            for b in buckets:
                if b and len(chosen) < q:
                    chosen.append(b.pop())
        if len(chosen) < q:
            print(f"⚠️ {split}/{g}: only {len(chosen)}/{q} after {pages} pages; raise max_pages or adjust HOLDOUT_AFTER")
        picked += chosen
    print(f"{split}: planned {len(picked)} images from {pages} pages of metadata")
    return picked

def fetch_image(split, c, max_side):
    raw = http_get(c["src"], f"{split} image {c['row_idx']}", tries=2)
    if raw is None:                                 # image links are signed and expire after about a day
        page = rows_page(split, c["row_idx"], 1)
        raw = page and http_get(page["rows"][0]["row"]["image"]["src"], f"{split} image {c['row_idx']}")
    return to_png(raw, max_side) if raw else None

def sample(split, quotas, seed=SEED, max_side=MAX_SIDE, pool=3, max_pages=60, allow_partial=False):
    """Stratified, resumable sample of a split, downloading only the images it picks.

    The plan (which rows) and every downloaded image are checkpointed under SAMPLE_DIR,
    so after a network failure a rerun only downloads what's missing.
    """
    key = json.dumps([DATASET_ID, SUBSET, split, quotas, seed, max_side, pool, max_pages, HOLDOUT_AFTER])
    d = os.path.join(SAMPLE_DIR, f"{split}-{hashlib.md5(key.encode()).hexdigest()[:8]}")
    os.makedirs(d, exist_ok=True)
    plan_path = os.path.join(d, "plan.json")
    if os.path.exists(plan_path):
        with open(plan_path) as fh:
            picked = json.load(fh)
    else:
        picked = plan(split, quotas, random.Random(f"{split}-{seed}"), pool, max_pages)
        write_atomic(plan_path, json.dumps(picked).encode())

    def one(c):
        path = os.path.join(d, f"{c['row_idx']}.png")
        if not os.path.exists(path):
            png = fetch_image(split, c, max_side)
            if png is None:
                return None
            write_atomic(path, png)
        with open(path, "rb") as fh:
            return {"id": f"{split}-{c['row_idx']}", "split": split, "group": c["group"], "label": c["label"],
                    "generator": c["model"], "release_date": c["release_date"], "prompt": c["prompt"],
                    "png": fh.read()}

    with ThreadPoolExecutor(max_workers=8) as pool_:
        items = [it for it in pool_.map(one, picked) if it]
    failed = len(picked) - len(items)
    if failed:
        msg = (f"{split}: {failed}/{len(picked)} images still failed after retries. "
               f"Rerun this cell: the other {len(items)} are saved in {d} and won't download again.")
        if not allow_partial:
            raise RuntimeError(msg + " Or pass allow_partial=True to continue without them.")
        print("⚠️ " + msg)
    print(f"{split}: {len(items)} images; {ram()}")
    order = {g: k for k, g in enumerate(quotas)}
    return sorted(items, key=lambda t: order[t["group"]])

In [ ]:
# Resumable: if a download fails, rerun this cell; finished images load from SAMPLE_DIR.
TRAIN = sample("train", {"real": N_TRAIN_PER_CLASS, "fake_seen": N_TRAIN_PER_CLASS}, max_side=256)
CAL = sample("validation", {"real": N_CAL_PER_CLASS, "fake_seen": N_CAL_PER_CLASS})
TEST = sample("test", N_TEST)

from collections import Counter
print()
for name, items in [("train", TRAIN), ("cal", CAL), ("test", TEST)]:
    print(f"{name:5}", dict(Counter(t["group"] for t in items)))
print(ram())
print("\ntest generators:")
for (g, gen), n in sorted(Counter((t["group"], str(t["generator"])) for t in TEST).items()):
    print(f"  {g:12} {gen:30} {n}")

# "seen" means released before HOLDOUT_AFTER, not "present in TRAIN": check how much really overlaps.
train_gens = {t["generator"] for t in TRAIN}
print()
for g in ("fake_seen", "fake_unseen"):
    sub = [t for t in TEST if t["group"] == g]
    print(f"{g}: {sum(t['generator'] in train_gens for t in sub)}/{len(sub)} from a generator also in TRAIN")
for name, items in [("train", TRAIN), ("cal", CAL), ("test", TEST)]:
    print(f"real sources in {name}:", sorted({t["generator"] for t in items if t["label"] == "real"}))

> **🔵 Why the dataset viewer API, not `load_dataset(..., streaming=True)`?** OpenFake is ~3.4 TB, stored grouped by source (long runs of one generator), with full-size images. Streaming can only read forward, downloading every image it skips, and a shuffle buffer big enough to mix sources holds thousands of images in RAM, enough to crash a 12 GB Colab runtime. The viewer's `/rows` endpoint returns 100 rows of metadata per request from **any offset**, each with a link to its own image, so `sample()` reads metadata from random places, picks a balanced set, and downloads exactly those images (~100–200 KB each).

> **⚠️ Production reality —** the viewer serves images re-encoded as JPEG (a 1024×1024 Flux image came back at ~100 KB). Every image, real or fake, goes through that same encode, so the file format can't leak the label; but "original" here means "as the viewer serves it", not the generator's raw output, which is already a mild form of laundering.

> **🔵 Seeing timeouts or `HTTPError` warnings?** Every request is retried with backoff; if an image still fails, the cell stops with a message. Just **rerun it**: the plan and every finished image are saved in `SAMPLE_DIR`, so only the missing ones download again. An `HF_TOKEN` secret (a free Hugging Face read token) gives higher rate limits. On a free Colab runtime the disk is wiped on restart; mount Google Drive (see the comment in the data cell) to keep samples between sessions.

> **⚠️ Production reality —** OpenFake's splits are not drawn from one distribution. When we checked (2026-10), validation's real photos came from LAION and Pexels while test's came from ImageNet and DOCCI, and test's older generators were mostly different ones from validation's. So a threshold calibrated to 5% FPR on one source of real photos may not hold it on another, which is the deployment problem in miniature. Read the overlap printout above before trusting any FPR below.

> **🚩 Common mistake —** a detector that learns the *dataset* instead of the task. If real photos arrive as JPEGs and fakes as PNGs, or at different sizes, a classifier can score near-perfectly by reading the file format. Every image here is decoded and re-saved the same way (`to_png`), and the detector always sees a 256-pixel version, so format and size can't leak the label.

## Build the eval set: disclosures and labels

The test set needs answers for all three conditions.

**Condition 3 (disclosure)** we control: a quarter of the fakes get a visible "AI-generated" label, a quarter get an IPTC `DigitalSourceType = trainedAlgorithmicMedia` metadata mark (the machine-readable vocabulary provenance standards use), and half get nothing.

Images can be scored in two **variants**: `original`, and `laundered` (downscaled and recompressed to JPEG, the way content arrives after a screenshot or re-upload). Laundering keeps the visible label but **strips the metadata**, so a file that was disclosed at the source can be undisclosed by the time it's re-published. The gold label follows the published file, not its history. `VARIANTS` starts as `["original"]` to keep the first run cheap; add `"laundered"` for the robustness experiment, which doubles the VLM calls.

In [ ]:
XMP = ('<x:xmpmeta xmlns:x="adobe:ns:meta/"><rdf:RDF xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#">'
       '<rdf:Description xmlns:Iptc4xmpExt="http://iptc.org/std/Iptc4xmpExt/2008-02-29/" '
       'Iptc4xmpExt:DigitalSourceType="http://cv.iptc.org/newscodes/digitalsourcetype/trainedAlgorithmicMedia"/>'
       '</rdf:RDF></x:xmpmeta>')

def save_png(img, pnginfo=None):
    buf = io.BytesIO()
    img.save(buf, format="PNG", pnginfo=pnginfo)
    return buf.getvalue()

def with_metadata(png):
    info = PngInfo()
    info.add_itxt("XML:com.adobe.xmp", XMP)
    return save_png(Image.open(io.BytesIO(png)), info)

def with_visible_label(png, text="AI-generated"):
    img = Image.open(io.BytesIO(png)).convert("RGB")
    draw, size = ImageDraw.Draw(img), max(14, img.height // 18)
    try:
        font = ImageFont.load_default(size=size)
    except TypeError:                      # Pillow < 10.1
        font = ImageFont.load_default()
    x0, y0, x1, y1 = draw.textbbox((0, 0), text, font=font)
    pad = size // 3
    draw.rectangle([8, 8, 8 + x1 - x0 + 2 * pad, 8 + y1 - y0 + 2 * pad], fill="black")
    draw.text((8 + pad - x0, 8 + pad - y0), text, fill="white", font=font)
    return save_png(img)

def launder(raw, quality=60, scale=0.75):
    img = Image.open(io.BytesIO(raw)).convert("RGB")
    img = img.resize((max(1, int(img.width * scale)), max(1, int(img.height * scale))))
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=quality)   # JPEG re-encode drops the PNG metadata
    return buf.getvalue()

DISCLOSURES = ["none", "none", "visible", "metadata"]
for i, t in enumerate(t for t in TEST if t["label"] == "fake"):
    t["disclosure"] = DISCLOSURES[i % len(DISCLOSURES)]
for t in TEST:
    t.setdefault("disclosure", "none")

def published(item, variant):
    raw = item["png"]
    if item["disclosure"] == "visible":
        raw = with_visible_label(raw)
    elif item["disclosure"] == "metadata":
        raw = with_metadata(raw)
    return launder(raw) if variant == "laundered" else raw

def disclosed(item, variant):
    return item["disclosure"] == "visible" or (item["disclosure"] == "metadata" and variant == "original")

**Condition 2 (meaning)** comes from the prompt, not the pixels: a text model reads each image's prompt and answers "does this depict a real, identifiable entity or event?" and "is it evidently fiction or satire?". These are **silver labels**, an LLM's reading of the prompt rather than a human's. That's a deliberate trade for a case study; exercise 1 asks you to audit them, and in production you'd have humans label a sample and measure agreement ([04/02](../04-evals/02-llm-as-judge.ipynb)).

The VLM later answers the same two questions from the *image alone*, so the comparison is fair: the labeller sees the prompt, the system under test never does.

In [ ]:
CACHE = json.load(open(CACHE_PATH)) if os.path.exists(CACHE_PATH) else {}

def cached(key, fn):
    if key not in CACHE:
        CACHE[key] = fn()
        with open(CACHE_PATH, "w") as f:
            json.dump(CACHE, f)
    return CACHE[key]

def parse_json(text):
    m = re.search(r"\{.*\}", text or "", re.DOTALL)
    try:
        return json.loads(m.group()) if m else None
    except json.JSONDecodeError:
        return None

SILVER_PROMPT = (
    "You label the prompt or caption of an image for an EU AI Act Article 50(4) audit. "
    "Reply with ONLY a JSON object: "
    '{"real_entity": <true if the image depicts an existing, identifiable person, place, object or event, '
    "so that a realistic version could falsely appear authentic>, "
    '"evident_fiction": <true if the content is evidently artistic, satirical, fantastical or fictional>}'
)

def silver(item):
    resp = client.chat.completions.create(
        model=TEXT_MODEL, max_tokens=MAX_TOKENS, temperature=0,
        messages=[{"role": "system", "content": SILVER_PROMPT},
                  {"role": "user", "content": item["prompt"][:2000] or "(no prompt)"}])
    out = parse_json(resp.choices[0].message.content) or {}
    return {"real_entity": out.get("real_entity") is True, "evident_fiction": out.get("evident_fiction") is True}

for t in TEST:
    t.update(cached(f"silver|{TEXT_MODEL}|{t['id']}", lambda: silver(t)))

def gold_violation(item, variant):
    return (item["label"] == "fake" and item["real_entity"] and not item["evident_fiction"]
            and not disclosed(item, variant))

for v in VARIANTS:
    n = sum(gold_violation(t, v) for t in TEST)
    print(f"{v:9}: {n} gold violations out of {len(TEST)} images")
    if n < 10:
        print("   ⚠️ too few positives for stable numbers: fine for an end-to-end run, raise N_TEST_PER_GROUP for results")

## Approach D — provenance check

The cheapest check, and the only one that answers condition 3 from the file: is there a machine-readable AI mark? We look for the IPTC `trainedAlgorithmicMedia` term and for an embedded C2PA manifest (its JUMBF box). Presence of a C2PA box is not *verification*: that needs the signature checked against a trust list (exercise 6). And SynthID-style invisible watermarks can't be checked here at all: Google's image detector is proprietary.

In [ ]:
def provenance(raw):
    return {"ai_metadata": b"trainedAlgorithmicMedia" in raw,
            "c2pa_manifest": b"jumb" in raw and b"c2pa" in raw}

for v in VARIANTS:
    marked = [t for t in TEST if t["disclosure"] == "metadata"]
    found = sum(provenance(published(t, v))["ai_metadata"] for t in marked)
    print(f"{v:9}: metadata mark found on {found}/{len(marked)} images that were marked at the source")

## Approach A — a traditional detector, by transfer learning

Training an image classifier from scratch needs millions of images. Transfer learning reuses a model that already learned general visual features: freeze a pretrained **CLIP** image encoder, embed every training image, and fit a **logistic regression** on the embeddings (a *linear probe*, the recipe behind UnivFD). Here it's ViT-B/32 on `2 × N_TRAIN_PER_CLASS` images, which a CPU can do; the appendix scales it to ViT-L/14 on 4,000 images and adds a LoRA fine-tune.

Its threshold is set on the **calibration** split so that at most `TARGET_FPR` of real images get flagged, then frozen before it sees the test set.

In [ ]:
import numpy as np, torch
from sklearn.linear_model import LogisticRegression
from transformers import CLIPModel, CLIPProcessor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def detector_input(raw, side=256):
    # One fixed view for every detector: same size whatever the source, so size can't leak the label.
    img = Image.open(io.BytesIO(raw)).convert("RGB")
    img.thumbnail((side, side))
    return img

class ClipEncoder:
    def __init__(self, model_id):
        self.model = CLIPModel.from_pretrained(model_id).to(DEVICE).eval()
        self.proc = CLIPProcessor.from_pretrained(model_id)

    @torch.no_grad()
    def __call__(self, raws, bs=32):
        feats = []
        for i in range(0, len(raws), bs):
            x = self.proc(images=[detector_input(r) for r in raws[i:i + bs]], return_tensors="pt").to(DEVICE)
            f = self.model.get_image_features(**x)
            f = f if torch.is_tensor(f) else f.pooler_output
            feats.append(torch.nn.functional.normalize(f.float(), dim=-1).cpu())
        return torch.cat(feats).numpy()

def train_probe(encoder, train):
    X = encoder([t["png"] for t in train])
    y = np.array([t["label"] == "fake" for t in train], dtype=int)
    clf = LogisticRegression(max_iter=2000).fit(X, y)
    return lambda raws: clf.predict_proba(encoder(raws))[:, 1]

def threshold_at_fpr(scores, labels, target):
    reals = sorted((s for s, l in zip(scores, labels) if l == "real"), reverse=True)
    k = int(target * len(reals))                 # allowed false positives
    return reals[k] + 1e-9 if k < len(reals) else 0.0

DETECTORS, THRESHOLDS = {}, {}

def register(name, score_fn):
    """Add a detector: calibrate its threshold on CAL, then every report below includes it."""
    DETECTORS[name] = score_fn
    THRESHOLDS[name] = threshold_at_fpr(score_fn([c["png"] for c in CAL]), [c["label"] for c in CAL], TARGET_FPR)
    print(f"{name}: threshold {THRESHOLDS[name]:.3f} at calibration FPR ≤ {TARGET_FPR:.0%}")

t0 = time.time()
register("A_clip_b32_probe", train_probe(ClipEncoder("openai/clip-vit-base-patch32"), TRAIN))
print(f"trained + calibrated in {time.time() - t0:.0f}s on {DEVICE}")

## Approach B — a VLM, end to end

One call per image. The VLM answers every condition it can see: is it fake (`p_fake`), does it depict a real entity, is it evident fiction, does it carry a visible AI label. It cannot read file metadata, which is realistic: a VLM sees pixels. The `evidence` sentence is kept for the reviewer UI and **never scored**: a plausible reason is not a correct one.

The hybrid (C, below) reuses this same call, so B and C differ only in *who decides "is it fake"*: a clean ablation at no extra cost.

In [ ]:
VLM_PROMPT = (
    "You audit images under EU AI Act Article 50(4). Look at the image and reply with ONLY a JSON object:\n"
    '{"p_fake": <0-1, probability the image is AI-generated or AI-manipulated; ordinary compression, '
    "resizing or filters are NOT manipulation>,\n"
    ' "real_entity": <true if it depicts an existing, identifiable person, place, object or event '
    "and could falsely appear authentic>,\n"
    ' "evident_fiction": <true if it is evidently artistic, satirical, fantastical or fictional>,\n'
    ' "visible_ai_label": <true if the image itself shows a visible notice that it is AI-generated>,\n'
    ' "evidence": "<one short sentence>"}'
)

def data_uri(raw):
    mime = "image/png" if raw[:4] == b"\x89PNG" else "image/jpeg"
    return f"data:{mime};base64," + base64.b64encode(raw).decode()

def vlm_judge(raw):
    t0 = time.time()
    resp = client.chat.completions.create(
        model=VISION_MODEL, max_tokens=MAX_TOKENS, temperature=0,
        messages=[{"role": "user", "content": [
            {"type": "text", "text": VLM_PROMPT},
            {"type": "image_url", "image_url": {"url": data_uri(raw)}}]}])
    out = parse_json(resp.choices[0].message.content) or {}
    try:
        p = min(max(float(out["p_fake"]), 0.0), 1.0)
    except (KeyError, TypeError, ValueError):
        p = None                                  # unparseable: counted, not guessed
    return {"p_fake": p,
            **{k: out.get(k) is True for k in ("real_entity", "evident_fiction", "visible_ai_label")},
            "evidence": str(out.get("evidence", ""))[:200],
            "tokens": resp.usage.total_tokens if resp.usage else None,
            "latency": time.time() - t0}

> **⚠️ Production reality —** a generative model's `p_fake` is a *token it wrote*, not a calibrated probability: expect clustering at round numbers and confident, specific-sounding evidence on genuinely real photos. Published studies find the strongest multimodal models can be competitive zero-shot while many others sit near chance, so measure the one you'd ship rather than trusting a leaderboard.

## One image, step by step

Before scoring the whole test set, follow a single image through every approach. We pick a fake whose prompt names a real entity (a deepfake candidate) and publish it twice: once with no disclosure (a violation) and once with a visible "AI-generated" label (not one). Watch which approaches notice the difference. The four systems being compared:

| System | Flags when |
|---|---|
| **A only** | the detector says fake (the naive build: every fake is a "violation") |
| **A + D** | detector says fake **and** no metadata mark |
| **B end to end** | the VLM says fake, real entity, not fiction, no visible label |
| **C hybrid** | detector says fake **and** VLM says real entity, not fiction, no visible label **and** no metadata mark |

This costs at most two VLM calls, and it's also your first real measurement of tokens per image: the budget estimate below uses it.

In [ ]:
from IPython.display import display

SYSTEMS = ["A only", "A + D", "B end to end", "C hybrid"]

def flag(r, system, det):
    v = r["vlm"]
    is_fake = r["det"][det] >= THRESHOLDS[det]
    meaning = v["real_entity"] and not v["evident_fiction"] and not v["visible_ai_label"]
    if system == "A only":
        return is_fake
    if system == "A + D":
        return is_fake and not r["ai_metadata"]
    if system == "B end to end":
        return (v["p_fake"] or 0) >= 0.5 and meaning
    return is_fake and meaning and not r["ai_metadata"]       # C hybrid

DET = list(DETECTORS)[0]
fakes = [t for t in TEST if t["label"] == "fake"]
demo = next((t for t in fakes if t["real_entity"] and not t["evident_fiction"]), fakes[0])
print(f"{demo['id']}: {demo['label']} image from {demo['generator']} "
      f"(released {demo['release_date'] or 'n/a'}, group {demo['group']})")
print(f"prompt: {demo['prompt'][:200]!r}")
print(f"silver labels from the prompt: real_entity={demo['real_entity']}, evident_fiction={demo['evident_fiction']}\n")

for disclosure in ("none", "visible"):
    case = {**demo, "disclosure": disclosure}
    raw = published(case, "original")
    preview = Image.open(io.BytesIO(raw))
    preview.thumbnail((320, 320))
    display(preview)

    score = float(DETECTORS[DET]([raw])[0])
    same_as_test = disclosure == demo["disclosure"]          # reuse the test-set call when it's the same file
    key = f"vlm|{VISION_MODEL}|original|{demo['id']}" if same_as_test else f"vlm|{VISION_MODEL}|demo-{disclosure}|{demo['id']}"
    vlm = cached(key, lambda: vlm_judge(raw))
    prov = provenance(raw)
    row = {"det": {DET: score}, "vlm": vlm, **prov}

    print(f"published with disclosure = {disclosure!r}")
    print(f"  A detector  : score {score:.2f} vs threshold {THRESHOLDS[DET]:.2f} → "
          f"{'fake' if score >= THRESHOLDS[DET] else 'real'}")
    print(f"  B VLM       : " + json.dumps({k: vlm[k] for k in ("p_fake", "real_entity", "evident_fiction", "visible_ai_label")}))
    print(f"                evidence {vlm['evidence']!r}; {vlm['tokens']} tokens, {vlm['latency']:.1f}s")
    print(f"  D provenance: {prov}")
    print(f"  gold        : {'VIOLATION' if gold_violation(case, 'original') else 'no violation'}")
    for s in SYSTEMS:
        print(f"  {s:12}: {'FLAG' if flag(row, s, DET) else '-'}")
    print()

> **🔵 Interview signal —** walking one example through every component before running the batch is the cheapest bug-finder there is. If the VLM's JSON doesn't parse, the detector's score looks inverted, or the gold label surprises you, you find out here for two API calls, not after a day of rate-limited batch runs.

## Run the test set

Every test image, in every variant in `VARIANTS`, through every registered detector and the VLM. VLM results are cached on disk, so a rate limit or a later rerun (e.g. after the appendix registers new detectors) never repeats a call.

> **🔵 Hitting a rate limit?** The VLM is the only expensive step: one call per test image per variant. The estimate printed below uses the tokens actually measured in the walkthrough (until then it assumes ~2.3k per call, from Groq's per-image figure). Groq's listed base limits for the vision model are 8k tokens/minute and **200k tokens/day**. The per-minute limit is handled by the client's retries. The daily one can't be retried away, so when it hits, the cell **pauses cleanly**: every finished call is cached, the reports below cover only the images that have VLM results, and rerunning after the reset fills in the rest. Originals run first. For a multi-day run, keep `CACHE_PATH` on Drive (see the data cell).

In [ ]:
import groq

DET_LATENCY = {}

def vlm_budget():
    seen = [v["tokens"] for k, v in CACHE.items() if k.startswith("vlm|") and v and v.get("tokens")]
    per_call = sum(seen) / len(seen) if seen else 2_300          # measured once we have some calls
    todo = sum(f"vlm|{VISION_MODEL}|{v}|{t['id']}" not in CACHE for v in VARIANTS for t in TEST)
    days = todo * per_call / VLM_TOKENS_PER_DAY
    print(f"VLM: {todo} of {len(TEST) * len(VARIANTS)} calls still to make, ~{per_call:,.0f} tokens each "
          f"≈ {todo * per_call / 1e3:,.0f}k tokens ≈ {days:.1f} day(s) at {VLM_TOKENS_PER_DAY / 1e3:,.0f}k/day")

def run_all():
    rows, paused = [], False
    for variant in VARIANTS:
        raws = [published(t, variant) for t in TEST]
        det = {}
        for name, fn in DETECTORS.items():
            t0 = time.time()
            det[name] = fn(raws)
            DET_LATENCY[name] = (time.time() - t0) / len(raws)
        for i, (t, raw) in enumerate(zip(TEST, raws)):
            key = f"vlm|{VISION_MODEL}|{variant}|{t['id']}"
            vlm = CACHE.get(key)
            if vlm is None and not paused:
                try:
                    vlm = cached(key, lambda: vlm_judge(raw))
                except groq.RateLimitError as e:   # the client already retried; what's left is usually the daily cap
                    print(f"⏸ VLM paused at {variant} {i + 1}/{len(TEST)}: {str(e)[:200]}")
                    paused = True
            rows.append({"id": t["id"], "variant": variant, "group": t["group"], "label": t["label"],
                         "generator": t["generator"], "disclosure": t["disclosure"], "prompt": t["prompt"],
                         "gold": gold_violation(t, variant), "real_entity_gold": t["real_entity"],
                         **provenance(raw), "vlm": vlm,
                         "det": {name: float(s[i]) for name, s in det.items()}})
            if (i + 1) % 20 == 0:
                print(f"  {variant}: {i + 1}/{len(TEST)}")
    return rows

def collect():
    """Run everything; keep only rows with a VLM result, so every approach is compared on the same images."""
    rows = run_all()
    done = [r for r in rows if r["vlm"] is not None]
    for v in VARIANTS:
        print(f"{v:9}: VLM results for {sum(r['variant'] == v for r in done)}/{len(TEST)} images")
    if len(done) < len(rows):
        print("⏸ The reports below cover only those images. Rerun this cell after the limit resets to fill in the rest.")
    return done

vlm_budget()
RESULTS = collect()

## Verdict 1 — detection: is it AI-generated?

Condition 1 alone, judged threshold-free (AUC) and at each approach's operating point. The detector uses its calibrated threshold; the VLM uses 0.5, since it has no calibration step (exercise 3 gives it one). Read three columns closely: `recall unseen` (generators newer than anything in training), `FPR` (real people flagged), and how both move from `original` to `laundered`.

In [ ]:
def det_score(approach, r):
    return r["vlm"]["p_fake"] if approach == "B_vlm" else r["det"][approach]

def det_threshold(approach):
    return 0.5 if approach == "B_vlm" else THRESHOLDS[approach]

def auc(pairs):
    # P(random fake scores above random real); ties count half.
    fakes = [s for s, l in pairs if l == "fake"]
    reals = [s for s, l in pairs if l == "real"]
    if not fakes or not reals:
        return float("nan")
    return sum((f > g) + 0.5 * (f == g) for f in fakes for g in reals) / (len(fakes) * len(reals))

def rate(rs, approach):
    rs = [r for r in rs if det_score(approach, r) is not None]
    return sum(det_score(approach, r) >= det_threshold(approach) for r in rs) / max(len(rs), 1)

def detection_report():
    print(f"{'approach':22} {'variant':10} {'AUC':>5} {'recall seen':>12} {'recall unseen':>14} {'FPR':>5} {'unparsed':>9}")
    for a in list(DETECTORS) + ["B_vlm"]:
        for v in VARIANTS:
            rs = [r for r in RESULTS if r["variant"] == v]
            scored = [(det_score(a, r), r["label"]) for r in rs if det_score(a, r) is not None]
            by = lambda g: [r for r in rs if r["group"] == g]
            print(f"{a:22} {v:10} {auc(scored):>5.2f} {rate(by('fake_seen'), a):>12.0%} "
                  f"{rate(by('fake_unseen'), a):>14.0%} {rate(by('real'), a):>5.0%} {len(rs) - len(scored):>9}")

detection_report()

### Which generators get through?

The average hides where each approach breaks. Recall per generator, on the original variant:

In [ ]:
def per_generator(approaches=None):
    approaches = approaches or list(DETECTORS) + ["B_vlm"]
    rs = [r for r in RESULTS if r["variant"] == "original" and r["label"] == "fake"]
    print(f"{'generator':30} {'group':12} {'n':>3} " + " ".join(f"{a[:16]:>16}" for a in approaches))
    for gen, group in sorted({(str(r["generator"]), r["group"]) for r in rs}, key=lambda x: (x[1], x[0])):
        sub = [r for r in rs if (str(r["generator"]), r["group"]) == (gen, group)]
        print(f"{gen:30} {group:12} {len(sub):>3} " + " ".join(f"{rate(sub, a):>16.0%}" for a in approaches))

per_generator()

## Verdict 2 — compliance: is it an undisclosed deepfake?

Now the question the team asked, for the four systems from the walkthrough, each producing a single "flag for review" decision. Precision is "of what we sent to reviewers, how much was a real problem"; recall is "of the real problems, how many we caught".

In [ ]:
def prf(rs, system, det):
    tp = sum(r["gold"] and flag(r, system, det) for r in rs)
    fp = sum(not r["gold"] and flag(r, system, det) for r in rs)
    fn = sum(r["gold"] and not flag(r, system, det) for r in rs)
    neg = sum(not r["gold"] for r in rs)
    return tp / max(tp + fp, 1), tp / max(tp + fn, 1), fp / max(neg, 1), tp + fp

def compliance_report(det=None):
    det = det or list(DETECTORS)[0]
    print(f"detector used by A/C: {det}\n")
    print(f"{'system':14} {'variant':10} {'precision':>10} {'recall':>7} {'false-flag rate':>16} {'flagged':>8}")
    for s in SYSTEMS:
        for v in VARIANTS:
            p, rc, ffr, n = prf([r for r in RESULTS if r["variant"] == v], s, det)
            print(f"{s:14} {v:10} {p:>10.0%} {rc:>7.0%} {ffr:>16.0%} {n:>8}")

compliance_report()

### Why does each system miss? By disclosure

Split the original variant by how each fake was disclosed. "A only" should flag the disclosed fakes as violations (that's its whole failure); "A + D" should clear the metadata ones but not the visibly labelled ones; B and C should clear both, if the VLM reads the label.

In [ ]:
def by_disclosure(det=None):
    det = det or list(DETECTORS)[0]
    rs = [r for r in RESULTS if r["variant"] == "original"]
    subsets = [("real", [r for r in rs if r["label"] == "real"])] + [
        (f"fake, {d}", [r for r in rs if r["label"] == "fake" and r["disclosure"] == d])
        for d in ("none", "visible", "metadata")]
    print(f"{'subset':16} {'n':>3} {'gold':>5} " + " ".join(f"{s:>13}" for s in SYSTEMS))
    for name, sub in subsets:
        print(f"{name:16} {len(sub):>3} {sum(r['gold'] for r in sub):>5} "
              + " ".join(f"{sum(flag(r, s, det) for r in sub):>13}" for s in SYSTEMS))
    print("(cells = how many each system flagged)")

by_disclosure()

### Cost and latency

A pixel detector costs milliseconds and no tokens; every VLM image costs a request. At platform scale that difference is the design.

In [ ]:
vl = [r["vlm"] for r in RESULTS if r["vlm"]["tokens"]]
print(f"VLM: {np.mean([v['tokens'] for v in vl]):.0f} tokens and {np.mean([v['latency'] for v in vl]):.2f}s per image "
      f"(unparseable: {sum(r['vlm']['p_fake'] is None for r in RESULTS)}/{len(RESULTS)})")
for name, sec in DET_LATENCY.items():
    print(f"{name}: {sec * 1000:.0f} ms per image on {DEVICE}, 0 tokens")

> **🔵 Interview signal —** the hybrid also has a cost lever B lacks: run the cheap detector on everything and call the VLM **only** on images it flags. If the detector flags 5% of traffic, VLM spend drops ~20×. Say that number in the design review.

## Error analysis — read the hybrid's mistakes

Numbers say *how often*; the rows say *why*. For each mistake of C on the original variant: what the detector scored, what the VLM claimed, and the silver label it disagreed with.

In [ ]:
def mistakes(system="C hybrid", det=None, variant="original", limit=15):
    det = det or list(DETECTORS)[0]
    bad = [r for r in RESULTS if r["variant"] == variant and flag(r, system, det) != r["gold"]]
    print(f"{len(bad)} mistakes for {system} ({variant})\n")
    for r in bad[:limit]:
        v = r["vlm"]
        kind = "FALSE FLAG" if flag(r, system, det) else "MISS"
        print(f"[{kind}] {r['id']} {r['group']} gen={r['generator']} disclosure={r['disclosure']}")
        print(f"   det={r['det'][det]:.2f} (thr {THRESHOLDS[det]:.2f})  vlm: real_entity={v['real_entity']} "
              f"fiction={v['evident_fiction']} label={v['visible_ai_label']}  gold real_entity={r['real_entity_gold']}")
        print(f"   prompt: {r['prompt'][:110]!r}")
        print(f"   vlm evidence: {v['evidence']!r}")

mistakes()

> **🚩 Common mistake —** reading the false flags and fixing the prompt until they go away. Each fix is tuned on the test set, and the next number you report is a best case you won't see in production. Collect the patterns, change the system, then re-measure on a **fresh** sample (rerun `sample("test", ...)` with a different `seed`).

## Appendix (optional, GPU) — a real transfer-learning detector

Everything above ran on a CPU. This appendix needs a GPU runtime (**Runtime → Change runtime type → T4 GPU**), adds stronger versions of approach A, and plugs them into the same reports, reusing the cached VLM results (no new API calls):

- **A2 — bigger frozen backbone, more data:** CLIP ViT-L/14 linear probe on 4,000 images.
- **A3 — LoRA fine-tune:** train low-rank adapters inside the ViT-L/14 attention layers plus a classification head, so the encoder itself adapts to generator artifacts ([06](../06-adaptation/01-fine-tune-vs-rag-vs-prompt.ipynb) covers LoRA). Trained twice: plain, and with **JPEG augmentation** (half the training images laundered), to test whether robustness can be trained in.

Expect 20–40 minutes on a T4: a few minutes to sample 4,000 training images, then the two fine-tunes. On a CPU runtime the cells check for a GPU and do nothing.

In [ ]:
%pip install -q peft

In [ ]:
GPU = torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0) if GPU else "none (appendix cells will skip)")

N_TRAIN_GPU = 2000
BIG_ID = "openai/clip-vit-large-patch14"
if GPU:
    TRAIN_BIG = sample("train", {"real": N_TRAIN_GPU, "fake_seen": N_TRAIN_GPU},
                       seed=SEED + 1, max_side=256, pool=1.5, max_pages=400)
    register("A2_clip_l14_probe", train_probe(ClipEncoder(BIG_ID), TRAIN_BIG))

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import CLIPImageProcessor, CLIPVisionModel

class LoraDetector(torch.nn.Module):
    def __init__(self, model_id, r=8):
        super().__init__()
        base = CLIPVisionModel.from_pretrained(model_id)
        self.backbone = get_peft_model(base, LoraConfig(r=r, lora_alpha=2 * r, lora_dropout=0.1,
                                                        target_modules=["q_proj", "v_proj"]))
        self.head = torch.nn.Linear(base.config.hidden_size, 1)

    def forward(self, pixel_values):
        return self.head(self.backbone(pixel_values=pixel_values).pooler_output).squeeze(-1)

def train_lora(train, model_id=BIG_ID, epochs=1, bs=16, lr=1e-4, jpeg_aug=False):
    proc = CLIPImageProcessor.from_pretrained(model_id)
    model = LoraDetector(model_id).to(DEVICE)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    scaler = torch.amp.GradScaler("cuda")
    rng, data = random.Random(SEED), list(train)
    pixels = lambda raws: proc(images=[detector_input(r) for r in raws], return_tensors="pt")["pixel_values"].to(DEVICE)

    for epoch in range(epochs):
        rng.shuffle(data)
        model.train()
        losses = []
        for i in range(0, len(data), bs):
            batch = data[i:i + bs]
            raws = [launder(t["png"]) if jpeg_aug and rng.random() < 0.5 else t["png"] for t in batch]
            y = torch.tensor([t["label"] == "fake" for t in batch], dtype=torch.float32, device=DEVICE)
            with torch.autocast("cuda", dtype=torch.float16):
                loss = torch.nn.functional.binary_cross_entropy_with_logits(model(pixels(raws)).float(), y)
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            losses.append(loss.item())
        print(f"epoch {epoch + 1}: mean loss {np.mean(losses):.3f}")

    model.eval()

    @torch.no_grad()
    def score(raws, bs=32):
        out = []
        for i in range(0, len(raws), bs):
            with torch.autocast("cuda", dtype=torch.float16):
                out.append(torch.sigmoid(model(pixels(raws[i:i + bs])).float()).cpu())
        return torch.cat(out).numpy()
    return score

if GPU:
    register("A3_l14_lora", train_lora(TRAIN_BIG))
    register("A3_l14_lora_jpegaug", train_lora(TRAIN_BIG, jpeg_aug=True))

Re-run every report with the new detectors. The VLM rows come from the cache, so only the detectors do new work. Then rerun the compliance table with the best detector inside the hybrid.

In [ ]:
if GPU:
    RESULTS = collect()
    detection_report()
    per_generator()

In [ ]:
if GPU:
    v = VARIANTS[-1]                                        # the laundered variant, if you enabled it
    best = max(DETECTORS, key=lambda d: auc([(r["det"][d], r["label"]) for r in RESULTS if r["variant"] == v]))
    print(f"best detector on {v} AUC:", best, "\n")
    compliance_report(det=best)
    by_disclosure(det=best)

> **⚠️ Production reality —** whichever detector wins today is trained on today's generators. Independent 2026 benchmarks of open-source detectors find no single model wins across datasets, and the newest commercial generators defeat most of them. Plan for it: keep the "unseen generator" split as a standing regression test ([04/03](../04-evals/03-regression-evals.ipynb)), retrain on a schedule, and track model versions ([08/03](../08-operations/03-experiment-tracking-and-registry.ipynb)).

## What the numbers should tell you

Run it and check these against your tables, rather than taking them on faith:

- **Detection is the detector's job.** A trained model, even a frozen-feature probe, should beat the zero-shot VLM on `p_fake` AUC, and fine-tuning should help most on seen generators. Unseen generators and laundering are where every approach loses ground; JPEG augmentation is the cheap fix to test.
- **Compliance is not detection.** "A only" flags every disclosed fake, so its precision on the compliance task is capped by how many fakes are already labelled. Only systems that check provenance *and* meaning can clear them.
- **Metadata is fragile.** The laundered variant strips it, so "A + D" and "C" lose that signal exactly where reposted content lives. Visible labels and robust watermarks survive; metadata alone is not a disclosure strategy.
- **The VLM earns its place on meaning.** Real entity, satire, a visible label: these are reading tasks, which is where a VLM is the right tool, and where a pixel classifier has nothing to say.

> **⭐ Key takeaway —** decompose the legal question into the conditions it's made of, give each condition the cheapest tool that answers it well, and measure each piece and the whole. Here that's a detector for *is it fake*, provenance for *was it disclosed*, a VLM for *what does it depict*, and a human for *is this a violation*. A single model asked the whole question is easier to build and harder to trust.

## Exercises

1. **Audit the silver labels.** Hand-label `real_entity` and `evident_fiction` for 30 test prompts. What's your agreement with the text model ([04/02](../04-evals/02-llm-as-judge.ipynb))? Recompute the compliance table on only the items where you agree: do the rankings change?
2. **In the wild.** OpenFake also has a `reddit` subset of images collected from the platform. Swap it in as the test set (keep train and calibration as they are). Which approach degrades least?
3. **Calibrate the VLM.** Score the calibration split with `vlm_judge` and choose its threshold with `threshold_at_fpr`, like the detector's. Does B's recall at 5% FPR change your verdict? Also bucket its `p_fake` and plot the observed fake rate per bucket.
4. **Hybrid economics.** Change the hybrid so the VLM is called only when the detector flags. What share of images reach the VLM, what's the token saving, and what recall do you lose (if any)?
5. **False-flag fairness.** False flags land on real people. Hand-tag the real test images by apparent age group and skin tone and compute FPR per group. If one group is flagged more often, what would you change: training data, threshold, or review policy?
6. **Real C2PA.** Install the `c2pa-python` package and replace the byte check in `provenance` with actual manifest reading and signature validation. Test it on a C2PA-signed sample image from the C2PA project. What does "manifest present but signature invalid" mean for the compliance decision?
7. **The cost at scale.** At 5M uploads/month with 1% undisclosed deepfakes, compute monthly reviewer load (flags × 1 minute) and VLM spend (assume a real $/token from [01/04](../01-model-apis/04-context-and-caching.ipynb)) for B and for the gated hybrid from exercise 4. Write the one-sentence design-doc recommendation.